In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from src.data.functions import json_to_dict

In [2]:
top_dir = Path().absolute()
PARENT_FILE_PATH = top_dir / "models" / "2025-07-22 B7H4 SAM 1000L Model"
model_config = json_to_dict(Path(PARENT_FILE_PATH, "B7H4_model_parameters.json"))

a_matrix = np.array(model_config["a_matrix"])
b_matrix = np.array(model_config["b_matrix"])

In [3]:
def controllability_matrix(A, B):
    n = A.shape[0]
    C = B
    for i in range(1, n):
        C = np.hstack((C, np.linalg.matrix_power(A, i) @ B))
    return C

In [4]:
C = controllability_matrix(a_matrix, b_matrix)

rank = np.linalg.matrix_rank(C)
print("Rank of controllability matrix:", rank)

if rank == a_matrix.shape[0]:
    print("The system is controllable.")
else:
    print("The system is NOT controllable.")

Rank of controllability matrix: 5
The system is controllable.


In [5]:
U, S, Vt = np.linalg.svd(b_matrix)

print("Singular values:", S)
print("Input direction V[:, 0]:", Vt.T[:, 0])
print("Mapped state direction U[:, 0]:", U[:, 0])

Singular values: [0.35485172 0.35436419 0.3533784 ]
Input direction V[:, 0]: [-0.79762324  0.49400505 -0.34605804]
Mapped state direction U[:, 0]: [ 0.13116955 -0.15228976 -0.56937949 -0.79589285  0.04431635]


In [6]:
def check_stability(A, dt=None):
    lam = np.linalg.eigvals(A)
    if dt is None:
        # continuous
        unstable = [l for l in lam if np.real(l) > 0]
        borderline = [l for l in lam if np.real(l) > -1e-8 and np.real(l) <= 0]
        print("Eigenvalues (continuous):", lam)
        print("Unstable (Re>0):", unstable)
        print("Borderline (Re≈0):", borderline)
    else:
        # discrete (A is discrete-time)
        mags = np.abs(lam)
        unstable = [l for l,m in zip(lam,mags) if m > 1.0]
        borderline = [l for l,m in zip(lam,mags) if np.isclose(m,1.0,rtol=1e-6,atol=1e-8)]
        print("Eigenvalues (discrete):", lam)
        print("Unstable (|λ|>1):", unstable)
        print("Borderline (|λ|≈1):", borderline)
    return lam

eigens = check_stability(a_matrix)

Eigenvalues (continuous): [-3.10635248+0.j         -0.47780824+0.j         -0.15675833+0.j
 -0.12705683+0.01046575j -0.12705683-0.01046575j]
Unstable (Re>0): []
Borderline (Re≈0): []


In [7]:
import numpy as np
from numpy.linalg import matrix_rank, eigvals, inv, svd

def controllability_matrix(A,B):
    n = A.shape[0]
    mats = [np.linalg.matrix_power(A,i).dot(B) for i in range(n)]
    return np.hstack(mats)

def check_model(A,B,C=None,D=None, model_name="model"):
    n = A.shape[0]
    print(f"\n=== DIAGNOSTICS for {model_name} ===")
    # eigenvalues
    lam = eigvals(A)
    print("A eigenvalues:", lam)
    # controllability
    Ctrb = controllability_matrix(A,B)
    rank_ctrb = matrix_rank(Ctrb)
    print(f"Controllability matrix shape {Ctrb.shape}, rank = {rank_ctrb} (need {n})")
    # B conditioning
    u,s,vh = svd(B, full_matrices=False)
    print("B singular values:", s)
    print("B col norms:", np.linalg.norm(B, axis=0))
    # DC gain (continuous)
    if C is not None:
        try:
            Ainv = inv(-A)
            G0 = C.dot(Ainv).dot(B) + (D if D is not None else 0)
            print("DC gain C(-A)^{-1}B + D:\n", G0)
        except Exception as e:
            print("DC gain compute failed:", e)
    # modal controllability (participation of B in eigenmodes)
    try:
        lam, V = np.linalg.eig(A)
        Vinv = np.linalg.inv(V)
        # modal controllability measure per mode: norm of (Vinv * B) row
        modal_ctrl = [np.linalg.norm(Vinv[i,:].dot(B)) for i in range(n)]
        print("Modal controllability (per eigenmode):", modal_ctrl)
    except Exception as e:
        print("Modal controllability failed:", e)

# Example usage:
# check_model(A1,B1,C=C,D=D, model_name="first_model")
check_model(a_matrix,b_matrix,C=None,D=None, model_name="second_model")

condition_number = np.linalg.cond(np.array(b_matrix))
print("Condition number of the matrix:", condition_number)


=== DIAGNOSTICS for second_model ===
A eigenvalues: [-3.10635248+0.j         -0.47780824+0.j         -0.15675833+0.j
 -0.12705683+0.01046575j -0.12705683-0.01046575j]
Controllability matrix shape (5, 15), rank = 5 (need 5)
B singular values: [0.35485172 0.35436419 0.3533784 ]
B col norms: [0.35435888 0.35383562 0.35440113]
Modal controllability (per eigenmode): [np.float64(0.347958333351101), np.float64(1.1269621674612007), np.float64(15.981665130957657), np.float64(21.397722929747243), np.float64(21.39772292974723)]
Condition number of the matrix: 1.004169256004408


In [8]:
b_test = [
        [
            0.0007513694368404085,
            0.12065455093036734,
            0.07052244714756144
        ],
        [
            0.24900043838526284,
            0.4240720253876945,
            0.1489597742109416
        ],
        [
            0.07552134228410867,
            0.1439097111504612,
            0.05640528122108516
        ],
        [
            0.21297625577464546,
            0.31902893853829284,
            0.17029045332007522
        ],
        [
            -0.026884861435593686,
            -0.17628768223251623,
            -0.08451580902087807
        ]
    ]

np.array(b_test)

condition_number = np.linalg.cond(np.array(b_test))
print("Condition number of the matrix:", condition_number)

Condition number of the matrix: 16.086721167894247


In [9]:
def matrix_stability_cost(
    a_matrix,
    b_matrix,
    discrete=False,
    eig_margin=-1e-3,   # continuous: real(eig) <= eig_margin
    disc_margin=0.99,   # discrete: |eig| <= disc_margin
    b_target_norm=1.0,  # desired Frobenius norm for B
    weights=None,       # dict of penalty weights
):
    """
    Simplified stability and scaling penalties for A and B matrices.

    Returns
    -------
    total_cost : float
        Combined cost from penalties.
    details : dict
        Individual penalty values.
    """
    if weights is None:
        weights = dict(
            eig=1.0,
            b_scale=1.0,
            b_cond=1.0
        )

    cost = 0.0

    # ================================================================
    # 1. Stability penalty for A matrix
    # ================================================================
    eigs = np.linalg.eigvals(a_matrix)
    if discrete:
        # Penalize eigenvalues with magnitude > disc_margin
        excess = np.clip(np.abs(eigs) - disc_margin, a_min=0.0, a_max=None)
    else:
        # Penalize positive real parts above eig_margin
        excess = np.clip(np.real(eigs) - eig_margin, a_min=0.0, a_max=None)
    eig_penalty = np.sum(excess**2)
    cost += weights["eig"] * eig_penalty

    # ================================================================
    # 2. B matrix authority (scaling) penalty
    # ================================================================
    b_norm = np.linalg.norm(b_matrix, ord="fro")
    b_scale_pen = ((b_norm / b_target_norm) - 1.0) ** 2
    cost += weights["b_scale"] * b_scale_pen

    # ================================================================
    # 3. B matrix condition number penalty
    # ================================================================
    try:
        b_cond = np.linalg.cond(b_matrix)
        cond_pen = (np.log1p(b_cond)) ** 2  # small growth for large cond
    except np.linalg.LinAlgError:
        cond_pen = 1e3  # fallback if singular
    cost += weights["b_cond"] * cond_pen

    # ================================================================
    # Return total and breakdown
    # ================================================================
    return cost, {
        "eig_penalty": eig_penalty,
        "b_scale_pen": b_scale_pen,
        "cond_pen": cond_pen,
    }

matrix_stability_cost(a_matrix, b_matrix)

(np.float64(0.632734205534657),
 {'eig_penalty': np.float64(0.0),
  'b_scale_pen': np.float64(0.1493899549518466),
  'cond_pen': np.float64(0.4833442505828104)})